In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Policy gradient II: what weights each step?

## Where this session fits

> **The limitation we start from: the raw return is a very noisy weight, so the estimate is very noisy (high variance)**

- **The question:** what can weight each step instead of the raw return, without changing the expected gradient, or at a price we can state?

**Where this session fits (position)**

- **Position: Part B (policy gradient), session 2 of its 4 sessions**
    - B1 gave the theorem: $\nabla_\theta J$ is an average of terms $\nabla_\theta \log \pi_\theta$, each multiplied by a **weight** (a single number)
    - here *weight* means this number, **not** the network's parameters, which are always called parameters and written $\theta$
    - the raw return was the weight
    - "weighting the returns" and "weighting the gradient terms" are the **same thing**: the return *is* the number that multiplies the gradient term
    - this session fills slot 1 with a baseline and the advantage, and meets **REINFORCE**, the theorem estimated by sampling

**Where this session fits (continued)**

- **Slots** (B1's plan): this session fills slot 1, the weight, and slot 3, the baseline (part of the weight)
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

<img src="images/partB_arc_b2.png" width=62%>

**By the end you can**

1. say where the noise in the weight comes from
2. define a baseline and the advantage, and say why a baseline that does not depend on the action does not bias the gradient
3. say why the advantage compares one action with the state's average return
4. write the advantage form of the gradient, and read REINFORCE as the theorem estimated by sampling, with a baseline
5. place REINFORCE in the slot table, and say why a critic (a learned estimate of a state's value) is the next step


**Recall: the policy gradient theorem of B1**

> **The gradient of the objective is an average over episodes of score function times weight. B1's weight was a return; this session changes the weight.**

$$
\nabla_\theta J(\theta) = \Exp{\traj \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt}) \; \rlret_{0,\traj} \right]
$$
- the **score function** $\nabla_\theta \log \pi_\theta(\act \mid \state)$ is the gradient of the log-probability of the action taken
    - it is the direction in parameter space that makes that action more likely
- $\rlret_{0,\traj}$ is the weight: the return of the whole episode
    - **"weight" is the general name for the number that multiplies the score function**
    - B1's weight was the return; in this session the weight changes from method to method

**The policy we are weighting: a network on a one-hot state**

> **The FrozenLake policy is the smallest network: the state is one-hot encoded, one dense softmax layer outputs the four move probabilities, and $\theta$ is its $64$ parameters.**

<img src="images/policy_network_ohe.png" width=70%>

- the $16$ states are **categories**: cell $9$ is not "bigger" than cell $8$
    - a number as the input would impose an order, so each state gets its own input
    - **one-hot encoding:** a vector of $16$ numbers, a $1$ at the state's position and $0$ elsewhere
- one dense layer with $4$ outputs and a softmax: **multinomial logistic regression** on the one-hot state
- the logits are computed from $\theta$ and the state; they are not the parameters
- an update from one state changes only that state's $4$ parameters
- the rest of Part B holds for any network; FrozenLake only makes it small

# Variance returns: where the noise in the weight comes from

**Why this section?** Bias and variance were defined for the *targets* of Monte Carlo and TD. The same two words now describe the *weight* in the policy gradient.

> **The weight is unbiased but has high variance, for three reasons. Each reason has a fix, and the fixes are this session and the next.**

- recall (from sampling)
    - **bias:** the average estimate's distance from the truth
    - **variance:** how much the estimate fluctuates between samples

**The weight has high variance, for three reasons**

- the weight in the theorem, a sampled return, is **unbiased** (B1) but has **high variance**
    - on FrozenLake one episode's raw-return estimate is about 40 times the exact gradient in length
        - $0.45$ against $0.011$, over all $64$ parameters
        - B1's figure shows one entry, about 30 times
    1. **it is the absolute return, not the relative return**
        - only the *differences* between returns (relative to a typical one) carry signal
        - the shared part adds noise
    2. **it is a sum of random rewards, from one sample**
        - the return adds up the rewards of **all the later steps**, each one random
        - so its noise grows with the steps that remain
    3. **no per-step credit:** the same number weights every step, so with sparse rewards we cannot say *which* step deserves the credit


**The three fixes**

1. subtract something every action shares: a **baseline** (answers the first reason, the shared part)
2. give each step its own weight: **credit assignment**, the **advantage**
    - answers the third reason, no per-step credit
    - uses only the rewards that follow a step, or a critic (item 3)
3. replace part of the sampled return by an estimate from a **critic**
    - a critic is a learned network that estimates the expected return from a state, as in Part A
    - answers the second reason, a sum of noisy rewards from one sample
    - as a baseline it lowers variance with no bias
    - as a bootstrap it lowers variance much further, at the price of bias, as TD did
    - it also helps with credit
    - this session covers fixes 1 and 2; the critic is the next session


## Fix 1 of 3: a baseline, subtract what every action would get anyway

**Why this section?** The first reason: the weight contains a part shared by all actions.

> **A baseline $b(\state)$ depends on the state but not on the action, and is subtracted from the weight. The expected gradient is unchanged; a good baseline lowers the variance.**

<img src="images/fl_value_map.png" width=30%>

- a **baseline** $b(\state)$ is a number that depends on the state but **not on the action**
    - a natural choice is the average return from that state, its **state value** (the map: FrozenLake, exact)
    - states near the goal have higher values than the start, so a good baseline depends on the state
    - REINFORCE uses a cheap constant; a critic gives the state-dependent one (B3)
- the return relative to the baseline, $\advseq = \rlret - b(\state)$, is called the **advantage** (defined exactly on a later slide)
    - computed at each step $\tt$ of an episode $\traj$, it is written $\advseq_{\traj,\tt}$: one advantage **per step**


**Why a baseline does not bias the gradient**

> **A baseline leaves the expected gradient unchanged, provided it does not depend on the action.**

- subtracting a baseline changes the weight from $\rlret$ to $\rlret - b(\state)$
    - so the gradient gains one extra term, $-\,b(\state) \, \nabla_\theta \log \pi_\theta(\actseq \mid \state)$
- **the extra term averages to zero**, by the **zero-mean score lemma** of B1: the score function $\nabla_\theta \log \pi_\theta$ has zero mean (proved in B1)
    - the formula below is that extra term, up to its sign

$$
\E_{\actseq \sim \pi_\theta}\bigl[ \nabla_\theta \log \pi_\theta(\actseq \mid \state) \, b(\state) \bigr] = b(\state) \, \E\bigl[ \nabla_\theta \log \pi_\theta(\actseq \mid \state) \bigr] = 0
$$

- this needs $b$ **not to depend on the action**: it must be fixed before the action is chosen
    - then it comes out of the expectation
- inside an episode, condition on the history up to step $\tt$
    - then $b(\stateseq_\tt)$ is fixed and only $\actseq_\tt$ is random
    - so the lemma applies at every step
- so subtracting $b$ leaves the expected gradient unchanged
- what changes is the variance
- a good baseline cancels the part of the return that all actions share
- what remains is the return *relative to the other actions*, which is what separates good from bad
- a baseline helps most when all returns share a sign: the raw return then pushes up every action it samples, even the worst

**Why a baseline lowers the variance**

> **The mean of weight times score is unchanged, so the variance changes only through the size of the weight. A baseline near the typical return makes the weight small.**

- the variance of the estimate is its average *squared* size minus its squared mean
    - the mean is fixed (the previous slide), so only the squared size can change
- the squared size grows with the squared weight, $(\rlret - b)^2$ times the squared score
- example: returns of about $100 \pm 5$ in a state
    - with $b = 0$ the weight is about $100$, almost all of it shared by every action
    - with $b \approx 100$ the weight is about $\pm 5$: only what the action added
- **the best simple baseline is the state value $\statevalfun_\pi(\state)$, which makes the weight an advantage**
    - it is the policy's average of the action values, so the weight is positive for better-than-average actions and negative for worse


**Definition: the advantage, one action against the policy's average**

> **Definition. The *advantage* of an action in a state is the value of that action minus the value of the state: $\advseq = \actvalfun_\pi(\state, \act) - \statevalfun_\pi(\state)$. It is positive when the action beats the policy's average.**

- **a bare $\pi$ in $\actvalfun_\pi$ and $\statevalfun_\pi$ always means the current policy $\pi_\theta$**
- $\actvalfun_\pi(\state, \act)$ is the expected return when we take this **one action** $\act$ in $\state$, then follow $\pi$
- $\statevalfun_\pi(\state)$ is the expected return **of the state**, averaged over the policy's actions
    - $\statevalfun_\pi(\state) = \sum_\act \pi_\theta(\act \mid \state) \, \actvalfun_\pi(\state, \act)$
    - a **probability-weighted** average: actions the policy takes often count more
- the sampled $\rlret_\tt - b$ is a sample of $\actvalfun_\pi - b$, because $\E[\rlret_\tt \mid \state, \act] = \actvalfun_\pi(\state, \act)$
    - it is a sample of the exact advantage when $b = \statevalfun_\pi(\state)$
- caution: a baseline computed from the same sample as the weight depends slightly on the action, so it adds a small bias

**The advantage, drawn: one action's value against the policy's average**

> **The advantage of an action is its value minus the state value: positive for an action better than the policy's average, negative for a worse one.**

<img src="images/b2_advantage_bars.png" width=46%>

- three actions with action values $\actvalfun_\pi = 9, 7, 5$ and probabilities $0.25, 0.5, 0.25$
- state value: $\statevalfun_\pi(\state) = 0.25 \cdot 9 + 0.5 \cdot 7 + 0.25 \cdot 5 = 7$
    - the weights are the policy's probabilities
    - an equal-weight average of $9, 7, 5$ is also $7$, but only because these numbers are symmetric
- the advantages are $+2$, $0$, $-2$: the weights when $b = \statevalfun_\pi(\state)$
    - the raw return would push every sampled action up
    - subtracting $b = 7$ pushes one action up, leaves one alone, and pushes one down
    - the expected gradient is unchanged (the zero-mean score lemma); what changes is the variance


## Fix 2 of 3: credit assignment, one weight per step

**Why this section?** The third reason (no per-step credit): one weight for a whole episode cannot say which step deserves the credit.

> **One weight per episode cannot say which step deserves the credit. The fix is an advantage per step.**

- the weight of B1 gives *every* step the same weight: the return of the whole episode
    - a language model writes a 100-word masterpiece and mistypes the last character: *all* steps are punished
- the fix: an **advantage per step**, $\advseq_{\traj,\tt}$
    - the same quantity as before, the return relative to a baseline, but computed separately for each step $\tt$, not once per episode
    - first ingredient: weight step $\tt$ by the rewards that *follow* it (next slide)
    - a **critic** assigns credit even with one end reward (next session)
- a baseline $b(\state)$ alone reduces noise but does not assign credit

**First ingredient: weight step $\tt$ by the return from step $\tt$ on**

> **B1 weights every step by the return of the whole episode. Weighting step $\tt$ only by the rewards after it leaves the expected gradient unchanged and removes noise.**

- B1's weight is $\rlret_{0,\traj}$: the rewards before step $\tt$ plus the rewards from step $\tt$ on
- the new weight is $\rlret_{\tt,\traj}$, the return of Part A: only the rewards from step $\tt$ on
    - we call it the *return-to-go* only to contrast it with the whole-episode return
- **why dropping the early rewards loses nothing on average**
    - the rewards before step $\tt$ are already fixed when action $\tt$ is chosen
    - so their sum is a quantity fixed before the action, and the zero-mean lemma of B1 applies to it
    - its contribution to the expected gradient is $0$ (proved in the B1 appendix)
- **why it lowers the noise**
    - the early rewards add random terms to the weight and carry no information about action $\tt$
- this is a different cut from the baseline's
    - the return from step $\tt$ drops the **rewards before the action**
    - the baseline drops a **shared offset**
- caution: with $\disc < 1$ the exact weight also carries a factor $\disc^\tt$
    - implementations drop it, which changes the objective (B1 appendix)


**Credit assignment, drawn: one episode, two choices of weight**

> **With the whole-episode return every step gets the same weight. With the return-to-go a step is credited only with the rewards that follow it.**

<img src="images/b2_credit_per_step.png" width=46%>

- toy episode: six steps, reward $+1$ after step 1 and $+3$ after step 4, with $\disc = 1$
- step 5 comes after every reward, so its return is $0$
- with a reward only at the end, every return is equal: per-step credit needs a critic (next session)


**What the advantage does to the gradient**

<table>
<thead><tr><th nowrap><div align="left">Advantage</div></th><th nowrap><div align="left">Effect on the gradient</div></th><th nowrap><div align="left">Result</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left"><b>positive</b> ($\advseq_{\traj,\tt} > 0$)</div></td><td nowrap><div align="left">positive multiplier</div></td><td nowrap><div align="left"><b>accelerate</b>: better than usual, make it more likely</div></td></tr>
<tr><td nowrap><div align="left"><b>negative</b> ($\advseq_{\traj,\tt} < 0$)</div></td><td nowrap><div align="left">flips the sign</div></td><td nowrap><div align="left"><b>brake</b>: below expectation, make it less likely</div></td></tr>
<tr><td nowrap><div align="left"><b>about zero</b></div></td><td nowrap><div align="left">nullifies the gradient</div></td><td nowrap><div align="left"><b>maintain</b>: as expected, nothing to learn</div></td></tr>
</tbody></table>

- advantage-based methods are the basis of modern policy gradient
- one well-known early example is A3C (Mnih et al., 2016)


**What the fixes bought on FrozenLake**

> **On FrozenLake the baseline and the return from step $\tt$ cut the spread only a little. The big step is a critic, in the next session.**

<img src="images/fl_variance_staircase.png" width=80%>

- **spread** here means the standard deviation (the square root of the variance) of a one-episode estimate over all 64 parameters, uniform policy
- the first four are unbiased
- the TD error (next session) trades some bias for a smaller spread
- the return from step $\tt$ differs from the whole-episode return only by the $0.01$ step costs of B1, since the reward comes at the end
- the exact gradient has length $0.011$: one episode's estimate is still about 30 times too large


## The unified form: one template, and each method fills in the advantage

**Why this section?** Part B will meet several methods. One formula holds them all, and shows what each method changes.

> **Every method of Part B is this one formula. A method is mostly a choice of how to compute the advantage $\advseq$.**

$$
\nabla_\theta J(\theta) = \E_{\traj \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt}) \; \advseq_{\traj,\tt} \right]
$$

- read the formula as a **template**, with the weight $\advseq_{\traj,\tt}$ of slot 1 as the part each method fills in
    - **from here on, the weight is an advantage, or an estimate of one**: the raw return of B1 becomes $\advseq$
- the formula is the theorem of B1 with one change: the raw return is replaced by an advantage

**One template, many methods: how each one fills in the advantage**

> **To meet a new method, ask three questions: how is the advantage computed, how far does one update move, and where does the baseline come from?**

- **REINFORCE** (this session): the return minus a baseline from past episodes
- **actor-critic** (B3): the advantage estimated from a *learned* value function, a critic
- **GRPO** (Part C): the advantage relative to a group of completions, with no critic
- **PPO** (B4) is the exception that keeps the advantage
    - it changes slot 2, how far one update moves: a clipped update
    - its advantage comes from a critic, as in actor-critic
- the three questions are the three slots of B1, so the slot table at the end of this session lists the methods

# REINFORCE: the theorem, estimated by sampling

**Why this section?** The simplest method with an advantage, and the most direct use of the theorem. It is the reference point for what follows.

> **REINFORCE with baseline is the policy gradient theorem estimated from sampled episodes: weight each $\nabla_\theta \log \pi_\theta$ by the return from that step, minus a baseline.**

- Sutton and Barto's **REINFORCE** (section 13.3) uses the return from each step and no baseline
- **REINFORCE with baseline** (13.4) subtracts one
- B1's estimator, with the whole-episode return, is **REINFORCE (over-simplified)**
- from here, "REINFORCE" alone means the version with a baseline

- it is the sampled estimator of B1, with two changes: the weight is the return from step $\tt$, and a baseline is subtracted
    - B1's weight was the return of the whole episode
- the weight is $\advseq_{\traj,\tt} = \rlret_{\tt,\traj} - b$
    - $\rlret_{\tt,\traj}$ is the return from step $\tt$: the rewards that follow it
    - $b$ is a baseline, often a moving average of the returns of past episodes
- it needs **no critic**: nothing is learned except the policy
- "Monte Carlo" is the same sense as in sampling: the full return of a complete sampled episode

**The REINFORCE loop, drawn**

> **REINFORCE is the sampling loop of B1 with one new box: the baseline $b$, taken from past episodes, subtracted in step 3.**

<img src="images/b2_reinforce_loop.png" width=95%>

- the drawn loop is **textbook REINFORCE**: one episode per update
    - for practical reasons we later use a batch of episodes per update (last section of this session)
- C2 redraws this loop with box 3 changed: the baseline becomes a group mean

**The REINFORCE loop**

- repeat
    1. sample an episode $\traj$ by following $\pi_\theta$
    2. compute the return $\rlret_{\tt,\traj}$ of every step $\tt$
    3. set the weight $\advseq_{\traj,\tt} = \rlret_{\tt,\traj} - b$
    4. step: $\theta \leftarrow \theta + \alpha \sum_\tt \advseq_{\traj,\tt} \, \nabla_\theta \log \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt})$
    5. update the baseline $b$, after the step has used it
- the baseline used in steps 3 and 4 is its value *before* this episode updates it
    - then $b$ cannot depend on this episode's actions
    - this is the condition of the baseline slide: no bias

**The moving-average baseline, and a worked step**

- a common baseline: a **moving average** of the returns of the episodes so far
    - $b_{\text{new}} = (1 - \eta)\, b_{\text{old}} + \eta \, \rlret_{0,\traj}$, with a small **mixing rate** $\eta$ (not the learning rate $\alpha$, and not the weight of slot 1)
- a worked step, with $\eta = 0.1$ and $b = 5$
    - an episode returns $8$: the weight is $8 - 5 = +3$, so its actions are pushed *up*
    - then $b \leftarrow 0.9 \cdot 5 + 0.1 \cdot 8 = 5.3$
    - the next episode returns $4$: the weight is $4 - 5.3 = -1.3$, so its actions are pushed *down*
- when the only reward comes at the end and $\disc = 1$, every step has the same return
    - so every step gets the same weight: no per-step credit
    - per-step credit with an end-only reward needs a critic (next session)

**Where the baseline comes from: past episodes now, a group later**

> **Slot 3 is the source of the baseline. REINFORCE takes it from past episodes. A later method takes it from a *group*: several episodes sampled now, from the same start.**

<img src="images/b2_baseline_source.png" width=72%>

- a **group** is $G$ episodes sampled with the current policy, all from the same starting situation
    - for a language model: $G$ completions of one prompt (the right panel)
    - the baseline for each of them is the group's mean return
- left: the baseline lags the policy, because it averages returns from earlier versions of the policy
- right: the group mean is immediate, at the price of $G$ samples per start
- C2 builds the right-hand side


**The same loop for a language model: one training prompt at a time**

- for each training prompt $\x$
    - generate an output $\y$ from the policy $\pi_\theta$
    - its reward $\rew$ comes from an evaluator
    - the baseline is the current moving average $b$ (a constant for this episode)
    - gradient step on $\theta$ in the direction $(\rew - b)\, \nabla_\theta \log \pi_\theta(\y \mid \x)$
    - then update $b$ by the rule above

**REINFORCE on FrozenLake: ten runs**

<img src="images/fl_reinforce_curve.png" width=80%>

- these runs use the **practical form**: a batch of 8 episodes per update, not textbook REINFORCE's single episode (last section)
- $J(\theta)$ is computed exactly after each update
- both versions learn, slowly and unevenly: after 1,500 updates the mean return is $0.05$ (raw) and $0.08$ (with a moving baseline), against $J^* = 0.35$
- a few runs jump up and most crawl: the noise of one sampled return
- the two versions differ in the weight and the baseline together, and ten runs are few: the gap is only suggestive
- the next session replaces part of the sampled return by a learned estimate


**REINFORCE in the unified form: what each part gives, and what it costs**

> **Fill the template with REINFORCE's values. Each part brings something (green) and costs something (red).**

<img src="images/b2_reinforce_template.png" width=76%>

- orange: the weight on each score function (slot 1, with the baseline of slot 3)
- blue: the rest of the template



**Each cost points to a later session**

> **A *critic* is a second network that learns the value of a state: the return to expect from it. An action is judged by whether the outcome beat that expectation.**

<table>
<thead><tr><th nowrap><div align="left">The cost</div></th><th nowrap><div align="left">The fix</div></th><th nowrap><div align="left">Session</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">a baseline that ignores the state</div></td><td nowrap><div align="left">a state-dependent baseline, an estimate of $\statevalfun_\pi(\state)$: the first role of a critic</div></td><td nowrap><div align="left">3</div></td></tr>
<tr><td nowrap><div align="left">the noisy tail of one sampled return</div></td><td nowrap><div align="left">replace the tail by a critic's estimate: the second role of a critic</div></td><td nowrap><div align="left">3</div></td></tr>
<tr><td nowrap><div align="left">each batch is used once: after the update its episodes came from the old policy</div></td><td nowrap><div align="left">reuse the batch for several updates: a probability ratio corrects for the old policy, and clipping keeps the correction trustworthy</div></td><td nowrap><div align="left">4</div></td></tr>
<tr><td nowrap><div align="left">a critic is a second network to train</div></td><td nowrap><div align="left">a baseline from a group of episodes of the same prompt</div></td><td nowrap><div align="left">Part C</div></td></tr>
</tbody></table>


## The slot table: review and preview

**Why this section?** A review of the slots filled so far, and a preview of what the next sessions change.

<table>
<thead><tr><th nowrap><div align="left">Method</div></th><th nowrap><div align="left">Slot 1: weight $\advseq$</div></th><th nowrap><div align="left">Slot 2: how far</div></th><th nowrap><div align="left">Slot 3: baseline source</div></th></tr></thead>
<tbody>
<tr><td colspan="4"><div align="left"><b>Review: filled so far</b></div></td></tr>
<tr><td nowrap><div align="left">REINFORCE, over-simplified (B1)</div></td><td nowrap><div align="left">return, no baseline</div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left">none</div></td></tr>
<tr><td nowrap><div align="left"><b>REINFORCE with baseline</b> (this session)</div></td><td nowrap><div align="left">return minus baseline</div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left">moving average of returns</div></td></tr>
<tr><td colspan="4"><div align="left"><b>Preview: what the next sessions change</b></div></td></tr>
<tr><td nowrap><div align="left">actor-critic (B3)</div></td><td nowrap><div align="left"><b>TD error from a critic</b></div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left"><b>the critic</b></div></td></tr>
<tr><td nowrap><div align="left">PPO (B4)</div></td><td nowrap><div align="left">advantage from a critic</div></td><td nowrap><div align="left"><b>clipped ratio</b></div></td><td nowrap><div align="left">the critic</div></td></tr>
<tr><td nowrap><div align="left">GRPO (Part C)</div></td><td nowrap><div align="left">group-relative advantage</div></td><td nowrap><div align="left">clipped ratio</div></td><td nowrap><div align="left"><b>mean of a group of episodes</b></div></td></tr>
</tbody></table>

- in the preview rows, bold marks the cell that is new
- the last row, GRPO, takes its baseline from a group of samples (Part C)
- "step size $\alpha$" in slot 2 is the plain gradient step


## From textbook REINFORCE to the practical form

**Why this section?** The last row of the slot table, GRPO, builds on how REINFORCE is run in practice. This is the bridge to Part C.

> **Textbook REINFORCE updates after every episode, with a moving-average baseline. The practical form samples a batch of episodes per prompt and takes the baseline from the batch.**

- a **rollout** is one sampled episode (for a language model, one completion)
- a **batch** is the set of rollouts used for one update

<table>
<thead><tr><th nowrap><div align="left">Feature</div></th><th nowrap><div align="left">Theoretical REINFORCE</div></th><th nowrap><div align="left">Practical (batch)</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">rollouts per prompt</div></td><td nowrap><div align="left">1</div></td><td nowrap><div align="left">several (for example $8$)</div></td></tr>
<tr><td nowrap><div align="left">baseline</div></td><td nowrap><div align="left">global moving average (historical)</div></td><td nowrap><div align="left">group mean (immediate)</div></td></tr>
<tr><td nowrap><div align="left">variance</div></td><td nowrap><div align="left">very high (small learning rates are usual)</div></td><td nowrap><div align="left">reduced by sampling more</div></td></tr>
<tr><td nowrap><div align="left">update frequency</div></td><td nowrap><div align="left">every trajectory</div></td><td nowrap><div align="left">after a group of trajectories</div></td></tr>
<tr><td nowrap><div align="left">compute per update</div></td><td nowrap><div align="left">low</div></td><td nowrap><div align="left">high (heavy generation)</div></td></tr>
<tr><td nowrap><div align="left">analogy</div></td><td nowrap><div align="left">your lifetime golf average</div></td><td nowrap><div align="left">ten swings from the same spot</div></td></tr>
</tbody></table>


# What we established, and what it sets up

- **Established**
    - the raw return is an unbiased but very noisy weight
        - it contains the shared part
        - it is one sample
        - it gives all steps the same credit
    - a baseline that does not depend on the action leaves the expected gradient unchanged and reduces the variance (the zero-mean fact)
    - the **advantage** is the return of one action relative to the state's average return
    - the unified form is $\E[\sum \nabla_\theta \log \pi_\theta \; \advseq]$
    - **REINFORCE** is the theorem estimated by sampling, with a moving-average baseline and no critic
        - it is still noisy

**What it sets up**

- **Sets up**
    - **the limitation:** REINFORCE's weight is still one sampled return, so each step is noisy
    - next session: replace part of the sampled return by a learned estimate, a **critic**: actor-critic
    - then slot 2, how far one update may move (PPO), then in Part C slot 3 without a critic (GRPO)
